<a href="https://colab.research.google.com/github/953625244013-cell/RFM-CLV-Personalized-Email/blob/main/RFM_CLV_Personalized_Email_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("your_dataset.csv")
df.head()

rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

def get_promotion(segment):
    promotions = {
        "Champions": "20% OFF",
        "Loyal Customers": "15% OFF",
        "Potential Loyalists": "10% OFF",
        "At Risk": "25% OFF",
        "Lost Customers": "30% OFF"
    }
    return promotions.get(segment, "10% OFF")

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Create Total Amount
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

# Set analysis date
analysis_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Calculate RFM
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (analysis_date - x.max()).days,
    'CustomerID': 'count',
    'TotalAmount': 'sum'
})

# Rename columns
rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()

# Calculate RFM scores

rfm['R_Score'] = pd.qcut(
    rfm['Recency'],
    5,
    labels=[5, 4, 3, 2, 1]
)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm.head()

rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)

rfm.head()

def get_segment(row):
    r = int(row['R_Score'])
    f = int(row['F_Score'])
    m = int(row['M_Score'])

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r >= 3 and f >= 2:
        return "Potential Loyalists"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2:
        return "Lost Customers"
    else:
        return "Others"

rfm['Segment'] = rfm.apply(get_segment, axis=1)

rfm.head()

rfm['Segment'].value_counts()

rfm[['Recency', 'Frequency', 'Monetary', 'RFM_Score', 'Segment']].head(10)

# Calculate average purchase value
average_purchase_value = (
    rfm['Monetary'] / rfm['Frequency']
)

# Estimate purchase frequency
purchase_frequency = rfm['Frequency']

# Estimate customer lifespan
customer_lifespan = 1

# Calculate CLV
rfm['CLV'] = (
    average_purchase_value *
    purchase_frequency *
    customer_lifespan
)

rfm.head()

rfm[['Recency', 'Frequency', 'Monetary', 'CLV', 'Segment']].head(10)

def get_promotion(segment):
    promotions = {
        "Champions": "20% OFF",
        "Loyal Customers": "15% OFF",
        "Potential Loyalists": "10% OFF",
        "At Risk": "25% OFF",
        "Lost Customers": "30% OFF",
        "Others": "10% OFF"
    }
    
    return promotions.get(segment, "10% OFF")

rfm['Promotion'] = rfm['Segment'].apply(get_promotion)

rfm.head()

rfm[['RFM_Score', 'Segment', 'CLV', 'Promotion']].head(10)

def generate_email(customer_name, segment, clv, promotion):
    email = f"""
Hi {customer_name},

Thank you for being one of our valued {segment}!

Your estimated Customer Lifetime Value is ₹{clv:.2f}.
As a special offer, you can enjoy {promotion} on your next purchase.

We appreciate your continued support and look forward to serving you again.

Best Regards,
Customer Relationship Team
"""
    return email

email = generate_email(
    "Priya",
    "Loyal Customers",
    1200,
    "15% OFF"
)

print(email)

def create_prompt(customer_name, segment, clv, promotion):
    prompt = f"""
Create a short and friendly personalized marketing email.

Customer Name: {customer_name}
Customer Segment: {segment}
Customer Lifetime Value: {clv:.2f}
Promotion: {promotion}

Requirements:
- Address the customer by name.
- Mention the promotion.
- Keep the email friendly and professional.
- Do not mention RFM scores.
- Keep it under 100 words.
"""
    return prompt

prompt = create_prompt(
    "Priya",
    "Loyal Customers",
    1200,
    "15% OFF"
)

print(prompt)

!pip install -q openai

from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

print("OpenAI API connected successfully!")

def generate_llm_email(customer_name, segment, clv, promotion):

    prompt = f"""
Create a short and friendly personalized marketing email.

Customer Name: {customer_name}
Customer Segment: {segment}
Customer Lifetime Value: {clv:.2f}
Promotion: {promotion}

Requirements:
- Address the customer by name.
- Mention the promotion.
- Make the email relevant to the customer segment.
- Do not mention RFM scores.
- Keep it under 100 words.
"""

    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )

    return response.output_text

email = generate_llm_email(
    "Priya",
    "Loyal Customers",
    1200,
    "15% OFF"
)

print(email)

# Select the first customer
customer = rfm.iloc[0]

print("Customer ID:", customer.name)
print("Segment:", customer['Segment'])
print("CLV:", customer['CLV'])
print("Promotion:", customer['Promotion'])

email = generate_llm_email(
    str(customer.name),
    customer['Segment'],
    customer['CLV'],
    customer['Promotion']
)

print(email)

test_customers = rfm.head(3).copy()

test_customers['Personalized_Email'] = test_customers.apply(
    lambda row: generate_llm_email(
        str(row.name),
        row['Segment'],
        row['CLV'],
        row['Promotion']
    ),
    axis=1
)

test_customers[['Segment', 'CLV', 'Promotion', 'Personalized_Email']]

final_output = test_customers.reset_index()

final_output = final_output[
    ['CustomerID', 'Recency', 'Frequency', 'Monetary',
     'RFM_Score', 'Segment', 'CLV', 'Promotion',
     'Personalized_Email']
]

final_output

final_output.to_csv(
    'RFM_CLV_Personalized_Emails.csv',
    index=False
)

print("File saved successfully!")

import matplotlib.pyplot as plt

segment_counts = rfm['Segment'].value_counts()

plt.figure(figsize=(8, 5))
segment_counts.plot(kind='bar')

plt.title('Customer Distribution by RFM Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

avg_clv = rfm.groupby('Segment')['CLV'].mean()

plt.figure(figsize=(8, 5))
avg_clv.plot(kind='bar')

plt.title('Average CLV by Customer Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Average CLV')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Final Insights

### RFM Analysis
RFM analysis was used to understand customer purchasing behavior based on:

- **Recency** – How recently the customer purchased
- **Frequency** – How frequently the customer purchased
- **Monetary** – How much the customer spent

### Customer Segmentation
Customers were divided into meaningful segments such as:

- Champions
- Loyal Customers
- Potential Loyalists
- At Risk
- Lost Customers

### CLV Analysis
Customer Lifetime Value (CLV) was calculated to estimate the value of each customer.

### Personalized Email Workflow
The customer segment, CLV, and selected promotion are used as inputs for the LLM.

The LLM generates a personalized marketing email for each customer.

### Complete Workflow

Customer Data
→ RFM Analysis
→ Customer Segmentation
→ CLV Calculation
→ Promotion Selection
→ LLM
→ Personalized Email

# Display final personalized email results

display(
    final_output[
        ['CustomerID', 'Segment', 'CLV', 'Promotion', 'Personalized_Email']
    ].head(3)
)

# Conclusion

This project uses RFM analysis and Customer Lifetime Value (CLV) to understand customer behavior and identify valuable customer segments.

Based on the RFM segments, suitable promotions are assigned to customers. These customer details, along with the segment, CLV, and promotion, are provided to an LLM to generate personalized marketing emails.

The workflow helps businesses understand customers, target them with relevant offers, and automate personalized email generation.

### Key Technologies Used

- Python
- Pandas
- NumPy
- Matplotlib
- RFM Analysis
- Customer Lifetime Value (CLV)
- OpenAI API
- LLM-based Email Generation
- Google Colab

# Project Architecture

Customer Transaction Data
        ↓
Data Preprocessing
        ↓
RFM Analysis
        ↓
RFM Scoring
        ↓
Customer Segmentation
        ↓
CLV Calculation
        ↓
Promotion Selection
        ↓
LLM Prompt Creation
        ↓
Personalized Email Generation
        ↓
Final Output